In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define path to the pre-processed SMARD dataset
file_path = (
    "/content/drive/MyDrive/Colab Notebooks/SMARD_Cleaned_20260806_20260816.csv"
)

# Verify file existence
if os.path.exists(file_path):
  print("Dataset found successfully! Proceeding with data loading...")
else:
  print(
      "File not found! Please verify the folder structure in your Google Drive."
  )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset found successfully! Proceeding with data loading...


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# ==========================================================
# 1. 15-MINUTE GERMAN SYSTEM IMBALANCE & REBAP ENGINE
# ==========================================================
np.random.seed(42)
N_DAYS = 30  # 30-day 15-minute simulation (2,880 quarter-hour periods)
N_PERIODS = N_DAYS * 24 * 4

timestamps = pd.date_range(
    start="2026-08-01 00:00:00", periods=N_PERIODS, freq="15min"
)
hour_of_day = timestamps.hour.to_numpy() + timestamps.minute.to_numpy() / 60.0

# Fundamental System Drivers: Load & Renewable Profiles
base_load_mw = 48000 + 16000 * np.clip(
    np.sin((hour_of_day - 5) * np.pi / 13), 0, 1
)
actual_load_mw = base_load_mw + np.random.normal(0, 1200, N_PERIODS)

solar_infeed_mw = 35000 * np.maximum(
    0, np.sin((hour_of_day - 5.5) * np.pi / 13)
)
wind_infeed_mw = (
    22000
    + 10000 * np.sin(np.linspace(0, 10 * np.pi, N_PERIODS))
    + np.random.normal(0, 1500, N_PERIODS)
)
total_renewables_mw = solar_infeed_mw + wind_infeed_mw
system_residual_load_mw = actual_load_mw - total_renewables_mw

# Day-Ahead Spot Reference Price (€/MWh)
da_price_eur = (
    40.0
    + (system_residual_load_mw / 45000) * 45.0
    + np.random.normal(0, 6, N_PERIODS)
)
da_price_eur = np.round(np.clip(da_price_eur, -30.0, 240.0), 2)

# Net Regulation Volume (NRV / Netzregelverbund in MW)
ramp_load = np.diff(actual_load_mw, prepend=actual_load_mw[0])
ramp_solar = np.diff(solar_infeed_mw, prepend=solar_infeed_mw[0])
nrv_mw = (
    -(ramp_load * 0.4) + (ramp_solar * 0.25) + np.random.normal(0, 1400, N_PERIODS)
)
nrv_mw = np.round(nrv_mw, 1)

# Target: 1 = System Short (Deficit, NRV < 0 -> High reBAP Risk), 0 = System Long (Surplus, NRV >= 0)
target_system_short = (nrv_mw < 0).astype(int)

# reBAP Pricing Engine according to German TSO / BNetzA logic
rebap_price_eur = np.zeros(N_PERIODS)
for i in range(N_PERIODS):
  base_p = da_price_eur[i]
  nrv = nrv_mw[i]
  if nrv < 0:
    deficit_factor = min(4.5, 1.0 + (abs(nrv) / 1800.0) ** 1.5)
    p = max(base_p * deficit_factor, base_p + abs(nrv) * 0.08) + np.random.normal(
        15, 20
    )
  else:
    surplus_factor = max(-0.8, 1.0 - (nrv / 1600.0) * 1.2)
    p = min(base_p * surplus_factor, base_p - nrv * 0.05) + np.random.normal(
        -5, 15
    )
  rebap_price_eur[i] = np.round(p, 2)

# ==========================================================
# 2. ML SYSTEM BALANCE STATE CLASSIFIER
# ==========================================================
df_features = pd.DataFrame({
    'hour': hour_of_day,
    'da_price': da_price_eur,
    'residual_load': system_residual_load_mw,
    're_share': total_renewables_mw / actual_load_mw,
    'ramp_load': ramp_load,
    'ramp_solar': ramp_solar,
    'load_mw': actual_load_mw,
})

split_idx = int(0.75 * N_PERIODS)
X_train, X_test = df_features.iloc[:split_idx], df_features.iloc[split_idx:]
y_train, y_test = (
    target_system_short[:split_idx],
    target_system_short[split_idx:],
)

clf = HistGradientBoostingClassifier(
    max_iter=150, learning_rate=0.08, random_state=42
)
clf.fit(X_train, y_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_proba)

# ==========================================================
# 3. BILANZKREIS FINANCIAL RISK & HEDGING ENGINE
# ==========================================================
test_len = len(y_test)
test_rebap = rebap_price_eur[split_idx:]
test_da = da_price_eur[split_idx:]
test_proba = y_pred_proba

# Simulated 15-min unhedged forecast deviations (MWh)
np.random.seed(99)
portfolio_imbalance_mwh = np.random.normal(0, 3.5, test_len)

# Strategy A: Unmanaged / Blind reBAP Settlement
loss_unmanaged_eur = np.where(
    portfolio_imbalance_mwh < 0,
    portfolio_imbalance_mwh * (test_rebap - test_da),
    portfolio_imbalance_mwh * (test_rebap - test_da),
)

# Strategy B: AI-Driven Risk-Hedging (Auto-Rebalancing in Intraday when Risk > 65%)
loss_hedged_eur = np.copy(loss_unmanaged_eur)
for j in range(test_len):
  p_short = test_proba[j]
  imb = portfolio_imbalance_mwh[j]
  if (p_short > 0.65 and imb < -1.0) or (p_short < 0.35 and imb > 1.0):
    loss_hedged_eur[j] = -abs(imb) * 8.0  # Intraday hedge execution spread

total_unmanaged_cost = -np.sum(loss_unmanaged_eur[loss_unmanaged_eur < 0])
total_hedged_cost = -np.sum(loss_hedged_eur[loss_hedged_eur < 0])
financial_savings = total_unmanaged_cost - total_hedged_cost

var_95_unmanaged = np.percentile(-loss_unmanaged_eur, 95)
var_95_hedged = np.percentile(-loss_hedged_eur, 95)

print('=' * 75)
print(' GERMAN REBAP IMBALANCE SETTLEMENT & SYSTEM BALANCE PREDICTION ENGINE')
print('=' * 75)
print(f'Test Horizon (15-min blocks):           {test_len} periods (180 hours)')
print(f'System Balance State ROC-AUC:           {auc_score:.3f}')
print(f'Unmanaged Bilanzkreis Imbalance Loss:   €{total_unmanaged_cost:,.2f}')
print(f'AI-Hedged Balancing Group Cost:         €{total_hedged_cost:,.2f}')
print(
    f'Net Financial Risk Savings:             +€{financial_savings:,.2f} (+'
    f'{(financial_savings/total_unmanaged_cost)*100:.1f}%)'
)
print(
    f'15-Min 95% Value-at-Risk (Unmanaged):   €{var_95_unmanaged:,.2f} /'
    ' quarter-hour'
)
print(
    f'15-Min 95% Value-at-Risk (AI-Hedged):   €{var_95_hedged:,.2f} /'
    f' quarter-hour (-{(1 - var_95_hedged/var_95_unmanaged)*100:.1f}%)'
)
print('=' * 75)

# ==========================================================
# 4. VISUALIZATION & EXPORT
# ==========================================================
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7.5))

# Plot 1: 3-Day Sample reBAP vs DA Price & Risk Probability
sample_test_slice = pd.DataFrame({
    'timestamp': timestamps[split_idx:],
    'da_price': test_da,
    'rebap_price': test_rebap,
    'pred_short_proba': test_proba,
}).iloc[96 * 2 : 96 * 5]

ax1.plot(
    sample_test_slice['timestamp'],
    sample_test_slice['rebap_price'],
    color='#DC2626',
    lw=1.8,
    label='reBAP Settlement Price (€/MWh)',
)
ax1.plot(
    sample_test_slice['timestamp'],
    sample_test_slice['da_price'],
    color='#1E293B',
    lw=1.5,
    linestyle='--',
    label='Day-Ahead Spot Reference (€/MWh)',
)
ax1.set_ylabel('Settlement Price [€/MWh]', fontweight='bold')
ax1.set_title(
    '15-Minute reBAP Settlement Price vs Day-Ahead Spot Spikes (3-Day Sample)',
    fontweight='bold',
    fontsize=11,
)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='upper right')

ax1_twin = ax1.twinx()
ax1_twin.plot(
    sample_test_slice['timestamp'],
    sample_test_slice['pred_short_proba'] * 100,
    color='#2563EB',
    lw=1.2,
    linestyle=':',
    label='P(System Deficit Short) %',
)
ax1_twin.axhline(
    65, color='#2563EB', linestyle='--', alpha=0.5, label='65% Risk Threshold'
)
ax1_twin.set_ylabel(
    'Model Risk Probability [%]', color='#2563EB', fontweight='bold'
)
ax1_twin.set_ylim(0, 110)

# Plot 2: Cumulative Penalty Loss Trajectory
cum_unmanaged = np.cumsum(np.maximum(0, -loss_unmanaged_eur)) / 1000.0
cum_hedged = np.cumsum(np.maximum(0, -loss_hedged_eur)) / 1000.0
days_test_x = np.arange(test_len) / (24 * 4)

ax2.plot(
    days_test_x,
    cum_unmanaged,
    color='#DC2626',
    lw=2.2,
    label='Unmanaged Imbalance Penalty (Full reBAP Exposure)',
)
ax2.plot(
    days_test_x,
    cum_hedged,
    color='#10B981',
    lw=2.5,
    label='AI-Driven Risk-Hedged Balancing Group',
)
ax2.set_title(
    'Cumulative Balancing Group (Bilanzkreis) Imbalance Penalties (Test Period)',
    fontweight='bold',
    fontsize=11,
)
ax2.set_xlabel('Test Horizon [Days]', fontweight='bold')
ax2.set_ylabel('Cumulative Penalty [Thousand €]', fontweight='bold')
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc='upper left')

plt.tight_layout()
plt.savefig('rebap_imbalance_settlement_prediction.png', dpi=300)
plt.show()

# Export Dataset
df_export_rebap = pd.DataFrame({
    'timestamp': timestamps[split_idx:],
    'da_price_eur': test_da,
    'rebap_price_eur': test_rebap,
    'pred_short_proba': np.round(test_proba, 4),
    'actual_system_short': y_test,
    'portfolio_imbalance_mwh': np.round(portfolio_imbalance_mwh, 2),
    'loss_unmanaged_eur': np.round(loss_unmanaged_eur, 2),
    'loss_hedged_eur': np.round(loss_hedged_eur, 2),
})
df_export_rebap.to_csv('rebap_imbalance_simulation_2026.csv', index=False)
print("Data exported as 'rebap_imbalance_simulation_2026.csv'")

 GERMAN REBAP IMBALANCE SETTLEMENT & SYSTEM BALANCE PREDICTION ENGINE
Test Horizon (15-min blocks):           720 periods (180 hours)
System Balance State ROC-AUC:           0.645
Unmanaged Bilanzkreis Imbalance Loss:   €94,833.48
AI-Hedged Balancing Group Cost:         €51,154.44
Net Financial Risk Savings:             +€43,679.04 (+46.1%)
15-Min 95% Value-at-Risk (Unmanaged):   €605.95 / quarter-hour
15-Min 95% Value-at-Risk (AI-Hedged):   €347.98 / quarter-hour (-42.6%)
